In [8]:

import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

# 1. Load dataset
data = pd.read_csv("english_malayalam_dataset.csv")

eng = data['english'].astype(str).str.lower()
mal = 'startseq ' + data['malayalam'].astype(str) + ' endseq'

# 2. Tokenization
et = Tokenizer(filters='')
mt = Tokenizer(filters='')

et.fit_on_texts(eng)
mt.fit_on_texts(mal)

# 3. Sequences and padding
X = pad_sequences(et.texts_to_sequences(eng), padding='post')
Y = pad_sequences(mt.texts_to_sequences(mal), padding='post')

# 4. Encoder
i = Input(shape=(X.shape[1],))
x = Embedding(len(et.word_index)+1, 64, mask_zero=True)(i)
_, h, c = LSTM(64, return_state=True)(x)

# 5. Decoder
d = Input(shape=(Y.shape[1]-1,))
y = Embedding(len(mt.word_index)+1, 64, mask_zero=True)(d)
y = LSTM(64, return_sequences=True)(y, initial_state=[h,c])
o = Dense(len(mt.word_index)+1, activation='softmax')(y)

# 6. Compile and train
model = Model([i,d], o)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit([X,Y[:,:-1]], Y[:,1:],
          sample_weight=(Y[:,1:] != 0),
          epochs=100, batch_size=8)



Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 4.0275
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3.9729 
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.6590
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.1977 
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.1061
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.0677
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.0285
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2.9852
Epoch 9/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.9391
Epoch 10/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.8803
Epoch 11/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2.8147
Epoch 12/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.7519
Epoch 13/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.6938
Epoch 14/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.6506
Epoch 15/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.60

In [13]:
 #7. Translation
s = input("Enter English sentence: ").lower()

x = pad_sequences(et.texts_to_sequences([s]),
                  maxlen=X.shape[1], padding='post')

token = [mt.word_index['startseq']]
words = {v:k for k,v in mt.word_index.items()}

for j in range(Y.shape[1]-1):

    d = pad_sequences([token], maxlen=Y.shape[1]-1, padding='post')
    p = model.predict([x,d], verbose=0)
    n = np.argmax(p[0,len(token)-1])

    if n == mt.word_index['endseq'] or n == 0:
        break

    token.append(n)

print("Malayalam:", ' '.join(words[j] for j in token[1:]))


Malayalam: നമസ്കാരം
